Utils and data_io Preprocessing

In [1]:
import pandas as pd
import os
import sys
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import re

# Notebook is in notebooks/, so repo root is parent
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
print(REPO_ROOT)

c:\Workspace\peak_detection\peak_detection


In [13]:
from peak_detection.models import PeakRange


def _get_top2_rrng_name(r: PeakRange) -> str:
    """Build an RRNG ion name from a PeakRange's top-two identification candidates."""
    detailed_id = r.detailed_id
    if detailed_id is None:
        return "Name:Unknown:0.0%-Unknown:0.0%"

    el1 = str(detailed_id.el1 or "Unknown")
    el2 = str(detailed_id.el2 or "Unknown")
    conf1 = float(detailed_id.conf1 or 0.0) * 100
    conf2 = float(detailed_id.conf2 or 0.0) * 100
    return f"Name:{el1}:{conf1:.0f}%-{el2}:{conf2:.0f}%"


def save_top2_rrng(filepath: str, detected_ranges: list[PeakRange], color_map: dict | None = None) -> None:
    """Write predicted ranges using top-two identifications in RRNG format."""
    ion_names = []
    seen_ion_names = set()

    for peak_range in detected_ranges:
        ion_name = _get_top2_rrng_name(peak_range)
        if ion_name not in seen_ion_names:
            ion_names.append(ion_name)
            seen_ion_names.add(ion_name)

    with open(filepath, 'w', encoding='utf-8') as f:
        f.write("[Ions]\n")
        f.write(f"Number={len(ion_names)}\n")
        for index, ion_name in enumerate(ion_names, 1):
            f.write(f"Ion{index}={ion_name}\n")
        f.write("\n")

        f.write("[Ranges]\n")
        f.write(f"Number={len(detected_ranges)}\n")
        for index, peak_range in enumerate(detected_ranges, 1):
            ion_name = _get_top2_rrng_name(peak_range)
            color_part = ""
            if color_map is not None:
                color = color_map.get(ion_name, "FF0000")
                color_part = f" Color:{color}"
            f.write(
                f"Range{index}={peak_range.start:.5f} {peak_range.end:.5f} "
                f"Vol:0.00000 {ion_name}{color_part}\n"
            )


In [14]:
from peak_detection.models import DetailedId

print(_get_top2_rrng_name(PeakRange(
    start=0,
    end=1,
    pos=0.5,
    label='ZrH',
    detailed_id=DetailedId(el1='ZrH', conf1=0.84, el2='Zr', conf2=0.12),
)))
print(_get_top2_rrng_name(PeakRange(
    start=0,
    end=1,
    pos=0.5,
    label='Unknown',
    is_unknown=True,
    detailed_id=DetailedId(el1='Unknown', conf1=0.61, el2='Zr', conf2=0.20),
)))
print(_get_top2_rrng_name(PeakRange(start=0, end=1, pos=0.5, label='Unknown', is_unknown=True)))


Name:ZrH:84%-Zr:12%
Name:Unknown:61%-Zr:20%
Name:Unknown:0.0%-Unknown:0.0%
